# Compute and plot land analysis-period bias metrics

Read model ensembles and references only for missing/forced products, cache metrics, and plot the cached maps.

## 1. Libraries


In [1]:
from pathlib import Path
import sys


## 2. User setup: paths


In [2]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_lnd_bias"


## 3. Project setup


In [3]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME, output_root=DIAGNOSTIC_OUTPUT_ROOT
)


## Notes: available variables, metrics, and periods

This notebook handles the **DA analysis period** only (`run_segment="da"`).
It is intentionally separate from the forecast-period workflow so their experiment
sets, periods, caches, and figures cannot be mixed.

Available land variables:

| Variable | Reference | Units |
|---|---|---|
| `TSA` | MODIS LST (`T2M`) | degC |
| `H2OSOI` | ESA CCI | m3 m-3 (model top 5 cm) |
| `SOILWATER_10CM` | CPC soil moisture | mm |

Available metrics:

| Metric | Meaning |
|---|---|
| `bias` | Ensemble-mean model minus reference, averaged over the selected period |
| `rmse` | Root-mean-square member error over ensemble and time |
| `spread` | Ensemble standard deviation averaged over time |
| `crps` | Continuous ranked probability score averaged over time |

Available experiment groups:

- `S0`: analysis period `201112`; forecast period `201201–201202`.
- `S1`: analysis period `201205`; forecast period `201206–201207`.

Only experiments with a configured `da` run are available. Use
`available_experiments(SEASON, RUN_SEGMENT)` after the imports to list them.
Each experiment/variable/period has a request-keyed NetCDF cache. Existing
products are reused unless `force_compute=True`. Set `N_BOOTSTRAP=0` to skip
confidence intervals; bias hatching is drawn only when cached intervals exist.


## 4. User setup: diagnostic selections


In [4]:
REFERENCE_DATA_ROOT = INPUT_DATA_ROOT / "reference"
CACHE_DIR = WORKFLOW_DATA_DIR / "bias_metrics"
FIGURE_DIR = WORKFLOW_FIGURE_DIR / "bias_metrics"

RUN_SEGMENT = "da"  # Fixed by this workflow: da=analysis, fc=forecast.
COMPONENT = "lnd"
EXPERIMENTS_BY_SEASON = {
    "S0": ["CTRL10-S0", "DART10-S0", "DART20-S0", "DART40-S0"],
    "S1": ["DART40-S1"],
}
SEASON = "S0"
EXPERIMENTS = EXPERIMENTS_BY_SEASON[SEASON]
SELECTED_VARIABLES = ["TSA", "H2OSOI", "SOILWATER_10CM"]
METRICS_TO_PLOT = ["bias", "rmse", "spread", "crps"]

LAT_RANGE = (-90, 90)
LON_RANGE = (-180, 180)
N_BOOTSTRAP = 100
RANDOM_SEED = 42
MAX_MEMBERS = None  # Set a small integer only for a quick test.

force_compute = False
RUN_WORKFLOW = False  # Review selections, then set True.


## 5. Workflow imports and availability


In [5]:
from util.bias_metrics import (
    METRICS,
    available_experiments,
    plot_cached_bias_metrics,
    run_bias_metrics,
)

print("Available experiments:", available_experiments(SEASON, RUN_SEGMENT))
print("Available metrics:", list(METRICS))


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


Available experiments: ['CTRL10-S0', 'DART10-S0', 'DART20-S0', 'DART40-S0']
Available metrics: ['bias', 'rmse', 'spread', 'crps']


## 6. Read, process, and cache


In [6]:
if RUN_WORKFLOW:
    run_bias_metrics(
        data_root=INPUT_DATA_ROOT,
        cache_dir=CACHE_DIR,
        season=SEASON,
        run_segment=RUN_SEGMENT,
        component=COMPONENT,
        experiments=EXPERIMENTS,
        variables=SELECTED_VARIABLES,
        reference_root=REFERENCE_DATA_ROOT,
        lat_range=LAT_RANGE,
        lon_range=LON_RANGE,
        n_bootstrap=N_BOOTSTRAP,
        random_seed=RANDOM_SEED,
        max_members=MAX_MEMBERS,
        force_compute=force_compute,
    )


## 7. Plot, save, and show


In [7]:
if RUN_WORKFLOW:
    figures = plot_cached_bias_metrics(
        data_root=INPUT_DATA_ROOT,
        cache_dir=CACHE_DIR,
        season=SEASON,
        run_segment=RUN_SEGMENT,
        component=COMPONENT,
        experiments=EXPERIMENTS,
        variables=SELECTED_VARIABLES,
        figure_dir=FIGURE_DIR,
        metrics=METRICS_TO_PLOT,
    )
    print(f"Wrote {len(figures)} figures to {FIGURE_DIR}")
